# 1. Kişi: Model Sorumlusu ve Vektörizasyon (Colab)
Bu notebook, NLP aşamasından çıkan temiz verileri alıp `all-MiniLM-L6-v2` modeli ile matematiksel vektörlere (embedding) çevirir.

In [ ]:
!pip install -q sentence-transformers pandas

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import pickle
import ast

# Colab'a dosya yüklemek için:
from google.colab import files
print("Lütfen 'cleaned_recipes.csv' dosyasını buraya yükleyin:")
uploaded = files.upload()

In [ ]:
# Veriyi okuma ve hazırlama
df = pd.read_csv('cleaned_recipes.csv')
print(f"Veri seti yüklendi. Toplam satır: {len(df)}")

def extract_text(val):
    try:
        lst = ast.literal_eval(val)
        return ' '.join(lst)
    except:
        return str(val)

sentences = df['ingredients_processed'].apply(extract_text).tolist()
print("İlk cümle örneği:", sentences[0])

In [ ]:
# Model Seçimi ve Yükleme
print("Model yükleniyor: all-MiniLM-L6-v2...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# GPU kullanımı otomatik algılanır
# Vektörizasyon Başlıyor
print("Vektörizasyon başlıyor, bu işlem GPU ile çok daha hızlı sürecektir...")
embeddings = model.encode(sentences, show_progress_bar=True)

print(f"Vektörizasyon tamamlandı! Matris boyutu: {embeddings.shape}")

In [ ]:
# Dışa Aktarma
output_file = 'recipe_embeddings.pkl'
with open(output_file, 'wb') as f:
    # Sadece embedding matrisi değil, hangi index'e ait olduklarını da tutuyoruz
    pickle.dump({'embeddings': embeddings, 'index': df.index.tolist()}, f)
    
print(f"{output_file} başarıyla kaydedildi!")

# Dosyayı bilgisayarınıza indirin
files.download(output_file)